# 🎬 تجارت‌یار — دفترچه‌ی ارائه روی Google Colab

این دفترچه برای **ارائه‌ی زنده** ساخته شده: یک سلول راه‌اندازی، یک سلول نمایش، یک سلول لینک عمومی،
یک سلول **تست سلامت قبل از ارائه**، و یک سلول **بازنشانی داده‌ی نمونه** برای وسط ارائه.

**روش اجرا:** از منوی `Runtime → Run all` استفاده کنید (یا سلول‌های ۱ تا ۳ را به‌ترتیب با `Ctrl+Enter`).
کل راه‌اندازی معمولاً **کمتر از ۲ دقیقه** طول می‌کشد (بدون `npm install` و بدون build — خروجی `dist` از قبل در مخزن است).

| سلول | کار | کی اجرا شود |
|---|---|---|
| ۱ | نصب Node + دریافت کد + اجرای سرور | قبل از ارائه |
| ۲ | نمایش برنامه داخل همین Colab | برای نمایش روی صفحه‌ی خودتان |
| ۳ | لینک عمومی موقت | اگر می‌خواهید لینک بدهید |
| ۴ | تست سلامت (۷ بررسی) | **۱۵ دقیقه قبل از ارائه** |
| ۵ | بازنشانی داده‌ی نمونه | وسط ارائه، اگر داده را به‌هم ریختید |
| ۶ | توقف / شروع مجدد | فقط در صورت خرابی |

### ✅ بدون هیچ تنظیمی اجرا کنید
سلول ۱ خودش شاخه‌ی درست را پیدا می‌کند: اول `main` را امتحان می‌کند و اگر آن نسخه «حالت ارائه» نداشت،
به شاخه‌ی دارای این قابلیت می‌رود. نام شاخه‌ی انتخاب‌شده را در خروجی چاپ می‌کند. **چیزی را دستی عوض نکنید.**

> ⚠️ **نکته‌ی مهم درباره‌ی داده‌ها:** سرور به‌صورت پیش‌فرض «خالی و صادق» بالا می‌آید.
> برای ارائه، `demo_data: True` در سلول ۱ فعال است تا کارتابل با **داده‌ی نمونه** پر شود؛
> در این حالت یک برچسب «داده نمونه — حالت ارائه» در هدر برنامه نمایش داده می‌شود تا با داده‌ی واقعی اشتباه نشود.

---
## بخش ۱ — راه‌اندازی

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  سلول ۱ — راه‌اندازی کامل: Node.js + دریافت کد + اجرای سرور
#  چند بار اجرا کردن این سلول بی‌خطر است (idempotent).
# ═══════════════════════════════════════════════════════════════════
import json, os, shutil, subprocess, time, urllib.request

CONFIG = {
    "repo":   "https://github.com/Setayesh-Jafari/Tejaratyarr.git",
    # به‌ترتیب امتحان می‌شوند؛ اولین شاخه‌ای که «حالت ارائه» را داشته باشد انتخاب می‌شود.
    # لازم نیست هیچ‌چیز را دستی عوض کنید.
    "branches": ["main", "arena/01a051da-tejaratyarr"],
    "dir":    "/content/Tejaratyarr",
    "port":   3000,
    "demo_data": True,          # ← داده‌ی نمونه برای ارائه (کارتابل خالی نماند)
    "reuse_existing_code": False,  # True = از کدِ قبلی استفاده کن، False = همیشه نسخه‌ی تازه از گیت
    "gemini_api_key": "",       # اختیاری: کلید Gemini برای فعال‌سازی AI واقعی
}
STATE = "/content/tj_state.json"
LOG   = "/content/tj_server.log"


def sh(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout)[-1500:])
    return r


def log(m):
    print(m, flush=True)


def http_json(path, timeout=4):
    try:
        with urllib.request.urlopen("http://localhost:%d%s" % (CONFIG["port"], path), timeout=timeout) as r:
            return json.loads(r.read().decode())
    except Exception:
        return None


# ─────────── ۱) Node.js ───────────
def node_major():
    if not shutil.which("node"):
        return 0
    v = sh("node -v").stdout.strip().lstrip("v")
    try:
        return int(v.split(".")[0])
    except ValueError:
        return 0

if node_major() < 18:
    log("⬇️  نصب Node.js 20 ...")
    sh("curl -fsSL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -o /tmp/node.txz")
    sh("tar -xJf /tmp/node.txz -C /usr/local --strip-components=1")
log("✅ Node.js: " + sh("node -v").stdout.strip())

# ─────────── ۲) کد پروژه ───────────
def supports_demo(dirpath):
    p = os.path.join(dirpath, "dist", "server.cjs")
    return os.path.isfile(p) and "SEED_DEMO" in open(p, encoding="utf-8", errors="ignore").read()


def clone(branch):
    sh("rm -rf " + CONFIG["dir"])
    sh("git clone --depth 1 --branch %s %s %s" % (branch, CONFIG["repo"], CONFIG["dir"]))


have_code = os.path.isfile(os.path.join(CONFIG["dir"], "dist", "server.cjs"))
chosen = None
if have_code and CONFIG["reuse_existing_code"]:
    chosen = "کدِ موجود روی دیسک"
    log("✅ " + chosen)
else:
    first_ok = None
    for b in CONFIG["branches"]:
        log("⬇️  دریافت کد از شاخه‌ی «%s» ..." % b)
        try:
            clone(b)
        except RuntimeError as e:
            log("   ⚠️  در دسترس نبود: %s" % str(e).strip().splitlines()[-1][:90])
            continue
        if first_ok is None:
            first_ok = b
        if supports_demo(CONFIG["dir"]):
            chosen = b
            break
        log("   ℹ️  این شاخه «حالت ارائه» ندارد؛ شاخه‌ی بعدی امتحان می‌شود.")
    if chosen is None:
        if first_ok is None:
            raise SystemExit("❌ هیچ شاخه‌ای کلون نشد — اتصال به GitHub را چک کنید.")
        clone(first_ok)
        chosen = first_ok
        log("⚠️  هیچ شاخه‌ای SEED_DEMO را پشتیبانی نمی‌کرد؛ از «%s» استفاده می‌شود (کارتابل خالی می‌ماند)." % first_ok)
log("✅ کد آماده است: %s  (شاخه: %s)" % (CONFIG["dir"], chosen))

# ─────────── ۳) سرور ───────────
if http_json("/api/health"):
    log("✅ سرور از قبل در حال اجراست (برای شروع مجدد، سلول ۶ و سپس دوباره سلول ۱)")
else:
    env = os.environ.copy()
    env["NODE_ENV"] = "production"
    env["PORT"] = str(CONFIG["port"])
    if CONFIG["demo_data"]:
        env["SEED_DEMO"] = "1"
    if CONFIG["gemini_api_key"]:
        env["GEMINI_API_KEY"] = CONFIG["gemini_api_key"]
    with open(LOG, "wb") as lf:
        proc = subprocess.Popen(
            ["node", "dist/server.cjs"], cwd=CONFIG["dir"], env=env,
            stdout=lf, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
            start_new_session=True,
        )
    json.dump(dict(CONFIG, branch=chosen, pid=proc.pid), open(STATE, "w"), ensure_ascii=False, indent=2)
    log("⏳  در حال بالا آمدن سرور ...")
    h = None
    for _ in range(40):
        if proc.poll() is not None:
            break
        h = http_json("/api/health")
        if h:
            break
        time.sleep(1)
    if not h:
        log("❌ سرور بالا نیامد. آخرین خطوط لاگ:")
        print(open(LOG, errors="ignore").read()[-2000:])
        raise SystemExit("سرور شروع نشد — لاگ بالا را بخوانید.")

# ─────────── ۴) گزارش وضعیت ───────────
h = http_json("/api/health")
boot = http_json("/api/bootstrap", timeout=10) or {}
inv, sup, dor = boot.get("inventory", []), boot.get("suppliers", []), boot.get("assessments", [])
log("")
log("═" * 52)
log("🎉  تجارت‌یار آماده‌ی ارائه است")
log("    آدرس محلی  : http://localhost:%d" % CONFIG["port"])
log("    نسخه       : %s  |  AI: %s" % (h.get("version"), ("فعال (" + str(h.get("model")) + ")") if h.get("aiEnabled") else "موتور محلی"))
log("    حالت ارائه : %s" % ("✅ داده‌ی نمونه فعال" if h.get("demoMode") else "❌ غیرفعال"))
log("    شاخه‌ی کد   : %s" % chosen)
log("    پرونده: %d  |  تأمین‌کننده: %d  |  ارزیابی: %d" % (len(inv), len(sup), len(dor)))
log("═" * 52)
if not h.get("demoMode") and len(inv) == 0:
    log("")
    log("⚠️  کارتابل خالی است: هیچ شاخه‌ی در دسترس، قابلیت SEED_DEMO نداشت.")
    log("    اگر داده‌ی نمونه نمی‌خواهید این طبیعی است؛ در غیر این صورت از دفترچه‌ی")
    log("    run_tejaratyar_colab_upload.ipynb با فایل Tejaratyarr.zip به‌روز استفاده کنید.")

---
## بخش ۲ — نمایش و لینک

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  سلول ۲ — نمایش برنامه داخل همین Colab (بدون تونل، بدون انتظار)
#  برای ارائه روی صفحه‌ی خودتان همین کافی است.
# ═══════════════════════════════════════════════════════════════════
import json
from google.colab import output

try:
    PORT = json.load(open("/content/tj_state.json"))["port"]
except Exception:
    PORT = 3000

print("🖥️  تجارت‌یار در پنجره‌ی زیر باز می‌شود (اگر خالی بود، یک‌بار Refresh بزنید):")
output.serve_kernel_port_as_window(PORT)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  سلول ۳ — لینک عمومی موقت (cloudflared) — فقط اگر لینک قابل ارسال می‌خواهید
#  لینک تا وقتی این Colab روشن بماند کار می‌کند.
#  اگر تونل برقرار نشد، سلول ۲ (نمایش داخل Colab) برای ارائه کافی است.
# ═══════════════════════════════════════════════════════════════════
import json, os, re, subprocess, time

CF, CFLOG = "/content/cloudflared", "/content/cloudflared.log"
try:
    PORT = json.load(open("/content/tj_state.json"))["port"]
except Exception:
    PORT = 3000


def fetch(url, dest, tries=2):
    for i in range(tries):
        r = subprocess.run(
            ["curl", "-fL", "--retry", "3", "--retry-all-errors", "--connect-timeout", "15", "-o", dest, url],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
            return True
        print("   تلاش %d ناموفق: %s" % (i + 1, (r.stderr or r.stdout).strip()[-160:]))
        time.sleep(2)
    return False


def main():
    if not (os.path.exists(CF) and os.path.getsize(CF) > 1_000_000):
        print("⬇️  دانلود cloudflared ...")
        ok = fetch("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
        if not ok:
            print("❌ دانلود cloudflared ممکن نشد (معمولاً محدودیت شبکه/VPN است).")
            print("   → برای ارائه روی صفحه‌ی خودتان از سلول ۲ (نمایش داخل Colab) استفاده کنید؛ به تونل نیاز ندارد.")
            print("   → اگر پروکسی/VPN دارید، روشنش کنید و همین سلول را دوباره اجرا کنید.")
            return
        os.chmod(CF, 0o755)

    subprocess.run("pkill -f 'cloudflared tunnel'", shell=True, capture_output=True)
    time.sleep(1)
    open(CFLOG, "w").close()
    subprocess.Popen(
        [CF, "tunnel", "--url", "http://localhost:%d" % PORT, "--no-autoupdate", "--logfile", CFLOG],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, stdin=subprocess.DEVNULL, start_new_session=True,
    )

    print("⏳  در انتظار لینک تونل (معمولاً ۱۰ تا ۳۰ ثانیه) ...")
    url = None
    for _ in range(60):
        try:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CFLOG, errors="ignore").read())
            if m:
                url = m.group(0)
                break
        except FileNotFoundError:
            pass
        time.sleep(2)

    print("")
    if url:
        print("═" * 52)
        print("🔗  لینک عمومی ارائه:")
        print("    " + url)
        print("═" * 52)
        print("نکته: لینک موقت است؛ تا وقتی این Colab روشن است کار می‌کند.")
    else:
        print("❌ لینک آماده نشد. دو راه دارید:")
        print("   ۱) همین سلول را دوباره اجرا کنید (اکثر مواقع بار دوم جواب می‌دهد).")
        print("   ۲) از سلول ۲ (نمایش داخل Colab) استفاده کنید — برای ارائه روی صفحه‌ی خودتان کافی است.")
        print("--- آخرین خطوط لاگ cloudflared ---")
        try:
            print(open(CFLOG, errors="ignore").read()[-1500:])
        except Exception as e:
            print(e)


main()

---
## بخش ۳ — اطمینان از سلامت (قبل از ارائه اجرا کنید)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  سلول ۴ — تست سلامت: همه‌ی مسیرهای حیاتی ارائه را یک‌جا بررسی می‌کند
#  ۱۵ دقیقه قبل از ارائه اجرا کنید؛ همه باید ✅ باشند.
# ═══════════════════════════════════════════════════════════════════
import json, re, time, urllib.request

try:
    PORT = json.load(open("/content/tj_state.json"))["port"]
except Exception:
    PORT = 3000
BASE = "http://localhost:%d" % PORT

def call(path, method="GET", body=None, timeout=15):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(BASE + path, data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read(), (time.time() - t0) * 1000
    except Exception as e:
        return getattr(e, "code", 0), str(e).encode(), (time.time() - t0) * 1000

results = []
def check(name, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        ok, detail = False, str(e)[:90]
    results.append(ok)
    print(("✅ " if ok else "❌ ") + name + " — " + str(detail))

def t_health():
    s, b, ms = call("/api/health")
    h = json.loads(b)
    return h.get("ok") is True, "نسخه %s | AI %s | حالت ارائه %s (%.0fms)" % (
        h.get("version"), "فعال" if h.get("aiEnabled") else "محلی",
        "فعال" if h.get("demoMode") else "غیرفعال", ms)

def t_bootstrap():
    s, b, ms = call("/api/bootstrap")
    d = json.loads(b)
    n = (len(d.get("inventory", [])), len(d.get("suppliers", [])), len(d.get("assessments", [])))
    return n[0] > 0, "پرونده %d | تأمین‌کننده %d | ارزیابی %d (%.0fms)" % (n[0], n[1], n[2], ms)

def t_index():
    s, b, ms = call("/")
    html = b.decode("utf-8", "ignore")
    ok = s == 200 and 'id="root"' in html
    return ok, "HTML صفحه‌ی اصلی %d بایت (%.0fms)" % (len(b), ms)

def t_assets():
    s, b, _ = call("/")
    html = b.decode("utf-8", "ignore")
    urls = re.findall(r'(?:src|href)="(/assets/[^"]+)"', html)
    if not urls:
        return False, "هیچ فایلی در index.html پیدا نشد"
    bad = []
    for u in urls:
        st, body, _ = call(u)
        if st != 200 or not body:
            bad.append("%s(%s)" % (u, st))
    return not bad, "%d فایل استاتیک بررسی شد%s" % (len(urls), "" if not bad else " — خراب: " + ", ".join(bad))

def t_ai():
    s, b, ms = call("/api/ai/hs-suggest", "POST", {"productName": "پنل خورشیدی مونوکریستال"})
    d = json.loads(b)
    return s == 200, "موتور %s | %d پیشنهاد (%.0fms)" % (d.get("engine"), len(d.get("suggestions", [])), ms)

def t_write():
    uid = "SMOKE-%d" % int(time.time())
    s1, b1, _ = call("/api/inventory", "POST",
                     {"id": uid, "name": "پرونده‌ی آزمایشی تست سلامت", "hsCode": "0000.00.00",
                      "status": "در گمرک (در حال ترخیص)", "stockQty": 1, "unit": "عدد"})
    s2, _, _ = call("/api/inventory/" + uid, "DELETE")
    return s1 == 201 and s2 == 200, "ایجاد %s و حذف %s (مسیر نوشتن سالم)" % (s1, s2)

def t_demo_reset():
    s, b, ms = call("/api/demo/seed", "POST")
    if s == 403:
        return True, "حالت ارائه فعال نیست (در ارائه‌ی واقعی اشکالی ندارد)"
    d = json.loads(b)
    c = d.get("counts", {})
    return c.get("inventory", 0) > 0, "بازنشانی داده نمونه: %d پرونده، %d تأمین‌کننده (%.0fms)" % (
        c.get("inventory", 0), c.get("suppliers", 0), ms)

print("🩺 تست سلامت تجارت‌یار — " + BASE)
print("─" * 52)
for name, fn in [("سرور و /api/health", t_health), ("داده‌ی کارتابل", t_bootstrap),
                 ("صفحه‌ی اصلی", t_index), ("فایل‌های استاتیک (dist)", t_assets),
                 ("موتور پیشنهاد تعرفه", t_ai), ("مسیر نوشتن (ایجاد/حذف)", t_write),
                 ("بازنشانی داده‌ی نمونه", t_demo_reset)]:
    check(name, fn)
print("─" * 52)
ok = sum(results)
print(("🎉 همه‌چیز سالم است (%d/%d) — آماده‌ی ارائه." % (ok, len(results))) if ok == len(results)
      else ("⚠️  %d از %d بررسی ناموفق بود — قبل از ارائه رفعش کنید." % (ok, len(results))))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  سلول ۵ — بازنشانی داده‌ی نمونه (وسط ارائه)
#  اگر هنگام دمو پرونده‌ای را حذف کردید یا وضعیت‌ها را به‌هم ریختید،
#  با این سلول در ~۱ ثانیه به حالت اول برمی‌گردید. بعد از آن صفحه را Refresh کنید.
# ═══════════════════════════════════════════════════════════════════
import json, urllib.request

try:
    PORT = json.load(open("/content/tj_state.json"))["port"]
except Exception:
    PORT = 3000

req = urllib.request.Request("http://localhost:%d/api/demo/seed" % PORT, method="POST",
                             headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=10) as r:
        d = json.loads(r.read().decode())
    c = d.get("counts", {})
    print("✅ داده‌ی نمونه بازنشانی شد: %d پرونده، %d تأمین‌کننده، %d ارزیابی"
          % (c.get("inventory", 0), c.get("suppliers", 0), c.get("assessments", 0)))
    print("🔄 حالا صفحه‌ی برنامه را Refresh کنید.")
except urllib.error.HTTPError as e:
    print("❌ خطای %d — %s" % (e.code, e.read().decode("utf-8", "ignore")))
    print("   اگر ۴۰۳ است یعنی سرور بدون SEED_DEMO=1 بالا آمده؛ در سلول ۱ مقدار demo_data را True بگذارید و دوباره اجرا کنید.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  سلول ۶ — توقف سرور (فقط در صورت خرابی)
#  بعد از اجرای این سلول، سلول ۱ را دوباره اجرا کنید.
# ═══════════════════════════════════════════════════════════════════
import json, os, signal, subprocess, time, urllib.request

try:
    st = json.load(open("/content/tj_state.json"))
except Exception:
    st = {}
PORT = st.get("port", 3000)


def alive():
    try:
        with urllib.request.urlopen("http://localhost:%d/api/health" % PORT, timeout=2) as r:
            return json.loads(r.read().decode()).get("ok") is True
    except Exception:
        return False


# ۱) کشتن تمیز با PID ثبت‌شده — فقط اگر آن PID واقعاً سرور ما باشد
pid = st.get("pid")
if pid:
    try:
        cmdline = open("/proc/%d/cmdline" % pid, "rb").read().decode("utf-8", "ignore")
        if "server.cjs" in cmdline:
            os.kill(pid, signal.SIGTERM)
    except (ProcessLookupError, FileNotFoundError, PermissionError):
        pass

# ۲) جاروی باقی‌مانده — الگوی [s] باعث می‌شود pkill شلِ خودش را نکشد
subprocess.run(r"pkill -f 'node dist/[s]erver\.cjs'", shell=True, capture_output=True)

for _ in range(10):
    if not alive():
        break
    time.sleep(0.5)

print("✅ سرور متوقف شد — حالا برای بالا آوردن دوباره، سلول ۱ را اجرا کنید."
      if not alive() else "⚠️ سرور هنوز پاسخ می‌دهد؛ همین سلول را دوباره اجرا کنید.")
print("")
print("آخرین خطوط لاگ سرور:")
try:
    print(open("/content/tj_server.log", errors="ignore").read()[-1200:])
except FileNotFoundError:
    print("(لاگی وجود ندارد)")

---
# 📋 سناریوی ارائه (۱۲ دقیقه)

**قانون طلایی:** اول لینک/پنجره‌ی برنامه را باز کنید، بعد حرف بزنید. اگر جایی گیر کردید، به بخش «نقشه‌ی نجات» پایین همین دفترچه بروید.

### ۰) پیش از شروع (۱۵ دقیقه قبل)
- سلول‌های ۱ تا ۳ را اجرا کنید؛ سپس **سلول ۴ (تست سلامت)** باید همه ✅ باشد.
- برنامه را یک‌بار باز کنید و روی هر ۵ بخش اصلی کلیک کنید تا فونت‌ها و نمودارها کشیده شوند.

### ۱) قلاب — ۱ دقیقه
> «واردکننده‌ی ایرانی امروز سه تصمیم پرهزینه می‌گیرد: چه کالایی، از چه تأمین‌کننده‌ای، با چه بهای تمام‌شده‌ای.
> تجارت‌یار این سه تصمیم را در یک کارتابل، با عدد و سند نشان می‌دهد.»

### ۲) نمای کلی + کارتابل کارگو — ۲ دقیقه
- صفحه‌ی «نمای کلی»: شاخص‌های سبد واردات و نرخ ارز.
- «کارتابل کارگو»: جستجو، فیلتر دسته، خروجی CSV.
- روی یک پرونده کلیک کنید → تایم‌لاین رویدادها و تغییر وضعیت.
- **جمله‌ی کلیدی:** «هر عددی که می‌بینید یا از داده‌ی ثبت‌شده می‌آید یا برچسب نمونه دارد.»

### ۳) گردش کار پرونده‌ها (کانبان) — ۱.۵ دقیقه
- جابه‌جایی یک پرونده بین مراحل + هشدار معطلی + ثبت یادداشت.

### ۴) ویزارد ارزیابی واردات — ۳ دقیقه (قلب ارائه)
- «ارزیابی جدید» → انتخاب سناریوی آماده (پنل خورشیدی) → پیش‌نمایش بهای تمام‌شده.
- تفکیک CIF، حقوق ورودی، سود بازرگانی، ارزش افزوده و هزینه‌های محلی.
- تحلیل حاشیه سود و حساسیت به نرخ ارز.
- **جمله‌ی کلیدی:** «فرمول‌ها همان قواعد گمرک ایران است، نه عدد دستی.»

### ۵) تفکیک تعرفه HS + اعتبارسنجی تأمین‌کننده — ۲ دقیقه
- جستجوی کالا → کد تعرفه، ماخذ، گروه صمت، و سناریوهای اختلافی.
- در اعتبارسنجی: امتیاز ریسک، مدارک، و «سطح اطمینان» منبع.
- صادق باشید: بدون کلید Gemini، موتور قاعده‌محور محلی کار می‌کند و UI هم همین را می‌گوید.

### ۶) داشبورد تحلیلی و دفتر مالی — ۱.۵ دقیقه
- نمودارها، بازارزیابی نرخ ارز، و اثر تغییر نرخ بر بهای تمام‌شده.

### ۷) جمع‌بندی — ۱ دقیقه
> «معماری: React 19 + TypeScript در فرانت، Express در بک‌اند، ذخیره‌سازی JSON اتمیک،
> و AI به‌صورت لایه‌ی افزایشی — اگر کلید نباشد، برنامه نمی‌خوابد.»

### پرسش‌های احتمالی و پاسخ کوتاه
| سؤال | پاسخ |
|---|---|
| داده‌ها واقعی است؟ | داده‌ی کارتابل نمونه است و برچسب دارد؛ قواعد محاسبه و دایرکتوری تعرفه واقعی‌اند. |
| چرا Colab؟ | برای دمو؛ استقرار واقعی روی هاست ایرانی (لیارا) با Dockerfile موجود در مخزن. |
| اگر Gemini نباشد؟ | موتور قاعده‌محور محلی جایگزین می‌شود و UI وضعیت را شفاف نشان می‌دهد. |
| مقیاس‌پذیری؟ | لایه‌ی ذخیره‌سازی برای ارتقا به SQLite طراحی شده؛ رابط توابع تغییر نمی‌کند. |

### 🆘 نقشه‌ی نجات
| مشکل | راه‌حل |
|---|---|
| کارتابل خالی است | سلول ۵ (بازنشانی داده‌ی نمونه) → Refresh |
| صفحه سفید است | Refresh؛ نشد سلول ۴ را اجرا کنید تا ببینید کدام فایل استاتیک خطا می‌دهد |
| لینک باز نمی‌شود | سلول ۳ را دوباره اجرا کنید؛ یا از پنجره‌ی سلول ۲ استفاده کنید |
| سرور خوابید | سلول ۶ و سپس سلول ۱ |
| اینترنت قطع شد | همه‌چیز داخل Colab است؛ فقط تونل می‌میرد → سلول ۲ (نمایش داخلی) |

> سناریوی کامل با زمان‌بندی دقیق در فایل `PRESENTATION.md` مخزن است.